<a href="https://colab.research.google.com/github/Orti-G/Thesis/blob/ML_Training/ThesisForecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

In [11]:
df = pd.read_csv("synthetic_household_training_data_v2.csv")
df.columns = df.columns.str.strip()  # clean any accidental spaces

In [12]:
print("Columns:", df.columns.tolist())
print("\nFirst value in timestamp column:")
print(df.iloc[0, 0])

Columns: ['Timestamp', 'Voltage (V)', 'Current (A)', 'Power (W)', 'Interval kWh', 'Cumul kWh', 'Hour', 'Day', 'Weekend', 'Note']

First value in timestamp column:
01/06/2025 00:00


In [14]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True)

print("Shape:", df.shape)
print("Range:", df["Timestamp"].min(), "→", df["Timestamp"].max())
print("Unique days:", df["Timestamp"].dt.date.nunique())

Shape: (8640, 10)
Range: 2025-06-01 00:00:00 → 2025-06-30 23:55:00
Unique days: 30


In [15]:
df["date"] = df["Timestamp"].dt.date

daily_totals = (
    df.groupby("date")["Interval kWh"]
    .sum()
    .reset_index()
    .rename(columns={"Interval kWh": "day_total_kWh"})
)

print(daily_totals.to_string(index=False))
print(f"\nMin  : {daily_totals['day_total_kWh'].min():.3f} kWh")
print(f"Max  : {daily_totals['day_total_kWh'].max():.3f} kWh")
print(f"Mean : {daily_totals['day_total_kWh'].mean():.3f} kWh")
print(f"Range: {daily_totals['day_total_kWh'].max() - daily_totals['day_total_kWh'].min():.3f} kWh")

      date  day_total_kWh
2025-06-01        36.4765
2025-06-02        22.8566
2025-06-03        22.9300
2025-06-04        23.5795
2025-06-05        26.1813
2025-06-06        21.4253
2025-06-07        31.4037
2025-06-08        33.5524
2025-06-09        21.1229
2025-06-10        22.8343
2025-06-11        24.1684
2025-06-12        21.2430
2025-06-13        21.9382
2025-06-14        31.1808
2025-06-15        32.2300
2025-06-16        22.3970
2025-06-17        21.2285
2025-06-18        23.0276
2025-06-19        24.1803
2025-06-20        20.6606
2025-06-21        38.4365
2025-06-22        29.2356
2025-06-23        22.0818
2025-06-24        24.0592
2025-06-25        22.6983
2025-06-26        23.9951
2025-06-27        20.3348
2025-06-28        33.9559
2025-06-29        28.1162
2025-06-30        20.9031

Min  : 20.335 kWh
Max  : 38.437 kWh
Mean : 25.614 kWh
Range: 18.102 kWh


In [17]:
# Step 3a — Engineer forecast features
df["frac_day_elapsed"] = (
    df["Timestamp"].dt.hour * 60 + df["Timestamp"].dt.minute
) / 1440.0

df["dow"] = df["Timestamp"].dt.dayofweek  # 0=Mon, 6=Sun

# Join daily target back to every interval row
df = df.merge(daily_totals, on="date")

print("Shape after merge:", df.shape)
print("\nSample at different times of day (Jun 1):")
sample = df[df["date"] == df["date"].iloc[0]]
sample = sample[sample["Timestamp"].dt.minute == 0]
sample[["Timestamp","Cumul kWh","frac_day_elapsed","Interval kWh","dow","Weekend","day_total_kWh"]].head(8)

Shape after merge: (8640, 14)

Sample at different times of day (Jun 1):


,Timestamp,Cumul kWh,frac_day_elapsed,Interval kWh,dow,Weekend,day_total_kWh
0,2025-06-01 00:00:00,0.0331,0.000000,0.0331,6,1,36.4765
12,2025-06-01 01:00:00,0.4885,0.041667,0.0430,6,1,36.4765
24,2025-06-01 02:00:00,0.9396,0.083333,0.0363,6,1,36.4765
36,2025-06-01 03:00:00,1.4072,0.125000,0.0340,6,1,36.4765
48,2025-06-01 04:00:00,1.8745,0.166667,0.0382,6,1,36.4765
60,2025-06-01 05:00:00,2.3167,0.208333,0.0363,6,1,36.4765
72,2025-06-01 06:00:00,2.7679,0.250000,0.0401,6,1,36.4765
84,2025-06-01 07:00:00,3.7852,0.291667,0.1479,6,1,36.4765


In [18]:
df

,Timestamp,Voltage (V),Current (A),Power (W),Interval kWh,Cumul kWh,Hour,Day,Weekend,Note,date,frac_day_elapsed,dow,day_total_kWh
0,2025-06-01 00:00:00,219.7,1.810,397.7,0.0331,0.0331,0,1,1,overnight_low,2025-06-01,0.000000,6,36.4765
1,2025-06-01 00:05:00,219.4,2.258,495.3,0.0413,0.0744,0,1,1,overnight_low,2025-06-01,0.003472,6,36.4765
2,2025-06-01 00:10:00,220.4,2.294,505.7,0.0421,0.1165,0,1,1,overnight_low,2025-06-01,0.006944,6,36.4765
3,2025-06-01 00:15:00,223.7,1.556,348.0,0.0290,0.1455,0,1,1,overnight_low,2025-06-01,0.010417,6,36.4765
4,2025-06-01 00:20:00,213.6,1.795,383.4,0.0319,0.1774,0,1,1,overnight_low,2025-06-01,0.013889,6,36.4765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8635,2025-06-30 23:35:00,220.2,1.801,396.6,0.0331,768.2927,23,30,0,normal,2025-06-30,0.982639,0,20.9031
8636,2025-06-30 23:40:00,221.1,1.700,375.8,0.0313,768.3240,23,30,0,normal,2025-06-30,0.986111,0,20.9031
8637,2025-06-30 23:45:00,219.7,1.858,408.2,0.0340,768.3580,23,30,0,normal,2025-06-30,0.989583,0,20.9031
8638,2025-06-30 23:50:00,221.0,2.099,463.9,0.0387,768.3967,23,30,0,normal,2025-06-30,0.993056,0,20.9031


In [19]:
# Step 3b — Define feature set and target
FEATURES = [
    "Cumul kWh",          # how much consumed so far today
    "frac_day_elapsed",   # how far into the day (0.0 = midnight, 1.0 = 23:55)
    "Interval kWh",       # current consumption rate
    "Hour",               # hour of day
    "dow",                # day of week
    "Weekend",            # weekend flag
]
TARGET = "day_total_kWh"

X = df[FEATURES]
y = df[TARGET]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature sample:")
X.head(3)

Feature matrix shape: (8640, 6)
Target shape: (8640,)

Feature sample:


,Cumul kWh,frac_day_elapsed,Interval kWh,Hour,dow,Weekend
0,0.0331,0.000000,0.0331,0,6,1
1,0.0744,0.003472,0.0413,0,6,1
2,0.1165,0.006944,0.0421,0,6,1


In [20]:
# Step 4 — Split last 4 days as test, rest as train
unique_dates = sorted(df["date"].unique())
cutoff_date  = unique_dates[-4]   # last 4 days = test

train = df[df["date"] < cutoff_date]
test  = df[df["date"] >= cutoff_date]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train: {len(train)} rows | {len(unique_dates)-4} days ({unique_dates[0]} → {unique_dates[-5]})")
print(f"Test : {len(test)} rows  | 4 days  ({cutoff_date} → {unique_dates[-1]})")
print(f"\nTrain target — min: {y_train.min():.3f}  max: {y_train.max():.3f}  mean: {y_train.mean():.3f}")
print(f"Test  target — min: {y_test.min():.3f}  max: {y_test.max():.3f}  mean: {y_test.mean():.3f}")

Train: 7488 rows | 26 days (2025-06-01 → 2025-06-26)
Test : 1152 rows  | 4 days  (2025-06-27 → 2025-06-30)

Train target — min: 20.661  max: 38.437  mean: 25.582
Test  target — min: 20.335  max: 33.956  mean: 25.827


In [21]:
# Step 5 — Train LR, RF, SVR
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
print("✓ Linear Regression trained")

# Random Forest
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("✓ Random Forest trained")

# Feature importances
fi = sorted(
    zip(FEATURES, rf.feature_importances_),
    key=lambda x: -x[1]
)
print("\n  Feature importances:")
for name, imp in fi:
    bar = "█" * int(imp * 40)
    print(f"  {name:<20} {bar} {imp:.3f}")

# SVR
svr = SVR(kernel="rbf", C=100, gamma="scale", epsilon=0.05)
svr.fit(X_train_sc, y_train)
print("\n✓ SVR trained")

✓ Linear Regression trained
✓ Random Forest trained

  Feature importances:
  dow                  ██████████████████ 0.459
  Weekend              ██████████████ 0.367
  Cumul kWh            █████ 0.143
  Interval kWh         █ 0.025
  frac_day_elapsed      0.005
  Hour                  0.002

✓ SVR trained


In [22]:
# Step 6 — Evaluate
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"  {name:<22}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")
    return dict(model=name, rmse=rmse, mae=mae, r2=r2)

print(f"  {'Model':<22}  {'RMSE':>10}  {'MAE':>10}  {'R²':>8}")
print("  " + "─"*56)

results = [
    evaluate("Linear Regression", y_test, lr.predict(X_test_sc)),
    evaluate("Random Forest",     y_test, rf.predict(X_test)),
    evaluate("SVR",               y_test, svr.predict(X_test_sc)),
]

# Pick winner
results_df = pd.DataFrame(results)
results_df["rank"] = (
    results_df["rmse"].rank() * 3 +
    results_df["mae"].rank()  * 2 +
    results_df["r2"].rank(ascending=False) * 1
)
best = results_df.loc[results_df["rank"].idxmin()]

print(f"\n🏆 WINNER: {best['model']}")
print(f"   RMSE = {best['rmse']:.4f} kWh")
print(f"   MAE  = {best['mae']:.4f} kWh")
print(f"   R²   = {best['r2']:.4f}")

  Model                         RMSE         MAE        R²
  ────────────────────────────────────────────────────────
  Linear Regression       RMSE=2.6568  MAE=2.2762  R²=0.7754
  Random Forest           RMSE=1.5201  MAE=1.4519  R²=0.9265
  SVR                     RMSE=4.2913  MAE=3.1273  R²=0.4141

🏆 WINNER: Random Forest
   RMSE = 1.5201 kWh
   MAE  = 1.4519 kWh
   R²   = 0.9265


In [23]:
os.makedirs("models", exist_ok=True)

forecast_payload = {
    "model":        rf,
    "scaler":       None,        # RF doesn't need scaling
    "feature_cols": FEATURES,
    "target_col":   TARGET,
    "model_name":   "Random Forest",
    "metrics": {
        "rmse": 1.5201,
        "mae":  1.4519,
        "r2":   0.9265,
    }
}

joblib.dump(forecast_payload, "models/forecast_model.pkl")
print("Saved → models/forecast_model.pkl")

# Verify it loads back correctly
loaded = joblib.load("models/forecast_model.pkl")
print("Model name :", loaded["model_name"])
print("Features   :", loaded["feature_cols"])
print("Metrics    :", loaded["metrics"])

Saved → models/forecast_model.pkl
Model name : Random Forest
Features   : ['Cumul kWh', 'frac_day_elapsed', 'Interval kWh', 'Hour', 'dow', 'Weekend']
Metrics    : {'rmse': 1.5201, 'mae': 1.4519, 'r2': 0.9265}
